# Why segmented predictions fail — with a routing audit of example 22

## Finding

**The roughly five-hour advance in 22 reproduces from the current hydrographs and is numerically stable under the current RAPID settings. It is an advance, not a five-hour total travel time.** Its magnitude is substantially influenced by the 5 m/s celerity cap. This validates the calculation within the model; it does not independently validate the model calibration or physical geometry against observations.

No frozen models, published responses, geometry inputs or calibration settings were changed. Isolated sensitivity runs live in a separate diagnostic directory.

### Direct response and numerical checks

- Re-read parent and segmented-child outlet files and independently recomputed centroid and global-maximum times on the shared windows.
- Primary centroid shift: −5.263388 h; peak shift: −5.716667 h. At 1%/2% window thresholds, centroid shifts are −5.122296/−4.895274 h; peak shifts remain unchanged.
- Re-routed both states, reproducing saved outputs. Halved native timestep and doubled routing cells in **both parent and child**: centroid shift changes by 0.192 seconds; peak shift changes by three minutes. Shared diagnostic baseline equals the published baseline.
- Verified flow fractions against geometry-based allocation, $\mathrm{count}\,K=L/c$, slope against boundary head drop, and all unchanged-reach parameters and visit probabilities between parent and child.

### Where does the advance originate?

Use the model diagnostic $T=\sum_e r_e L_e/c_e$. It is the mean route-time scale of this fixed linear routing configuration, not the finite-window centroid response and not a proposed hydrography-only predictor.

| Region in 22 | Parent route time | Child route time | Difference |
|---|---:|---:|---:|
| 14 collapsed blocks | 24.725 h | 17.594 h | −7.131 h |
| Unchanged connectors/tails | 9.897 h | 9.897 h | 0 h |
| Entire network | 34.622 h | 27.491 h | −7.131 h |

The unchanged slow reaches help explain why the segmented centerline is slower than the **old uniform centerline**. They do not generate the parent–segmented-child route-time difference: they cancel in that comparison.

Within each collapsed block, define $c_{\mathrm{eff},p}=L_p/T_p$. The exact algebraic decomposition is

$\Delta T=(L_c-L_p)/c_{\mathrm{eff},p}+L_c(1/c_c-1/c_{\mathrm{eff},p})$.

Summing over 22's blocks gives −2.247 h from the length term and −4.884 h from the speed term. This is a decomposition, not two independent physical interventions. The largest block contributions are CB002_bubble_11 (−1.804 h), CB001_bubble_9 (−1.002 h), and CB004_bubble_8 (−0.977 h).

A separate one-hour pulse, routed at the fixed parameters over a long diagnostic horizon, gives a parent–child pulse-centroid difference of approximately −7.124 h, close to the route-time diagnostic. The observed event is not an isolated pulse: initial/final equilibrium differ (19,150.44/18,840.56 m³/s), the common baseline is 18,825.53 m³/s, and its excess centroid is defined over a finite shared interval. Consequently its −5.263 h centroid shift need not equal −7.131 h. Pulse perturbations have small signed numerical undershoots and finite-horizon residuals; no clipping was applied, and pulse results do not replace the six response definitions.


For the focused parent–child explanation before predictor selection, see [why 22 has a modest collapse impact](hydrograph_22_collapse_impact.ipynb). Fourteen isolated block reruns reproduce almost the entire combined centroid advance: sum 5.20 h versus combined 5.26 h. The two largest blocks account for 67% of edited expected length but are already fast before collapse.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID').is_dir())
BASE=ROOT/'RAPID/outputs/hydrograph_closed_unit_extension_20261019'
OUT=BASE/'segmented_diagnostics'
display(pd.read_csv(OUT/'response22_recomputed.csv').round(6))
display(pd.read_csv(OUT/'refinement22.csv')[['state','variant','centroid_shift_hours','peak_shift_hours','maximum_discharge_difference_cms']])
display(pd.read_csv(OUT/'pulse22.csv'))

,window,recomputed_centroid_shift_hours,recomputed_peak_shift_hours
0,0.5%,-5.263388,-5.716667
1,1%,-5.122296,-5.716667
2,2%,-4.895274,-5.716667


,state,variant,centroid_shift_hours,peak_shift_hours,maximum_discharge_difference_cms
0,S000_base,current,0.000000,0.000000,0.000000
1,S000_base,refined,0.000000,0.000000,0.124971
2,all_closed_units_collapsed,current,-5.263388,-5.716667,0.000000
3,all_closed_units_collapsed,refined,-5.263335,-5.666667,0.248865


,state,expected_mean_route_hours,simulated_pulse_lag_hours,mass_ratio,minimum_perturbation_cms,final_perturbation_cms
0,S000_base,34.621821,34.618076,0.999998,-8.292409e-08,8.837778e-07
1,all_closed_units_collapsed,27.491127,27.494174,1.000002,-8.907122e-02,2.538072e-07


In [3]:
EXAMPLE='22'
m=pd.read_parquet(BASE/'state_metrics.parquet')
m=m[(m.example==EXAMPLE)&m.window.eq('0.5%')].set_index('state')
fig=go.Figure()
for state,label,color in [('S000_base','Parent','blue'),('all_closed_units_collapsed','Segmented child','red')]:
    r=m.loc[state];h=pd.read_parquet(r.outlet)
    fig.add_scatter(x=h.time_utc,y=h.q_outlet_cms,name=label,line=dict(color=color))
    fig.add_vline(x=float(r.peak_seconds)*1000,line_color=color,line_dash='dash')
parent_peak=pd.to_datetime(m.loc['S000_base'].peak_seconds,unit='s',utc=True)
fig.update_layout(title=f'{EXAMPLE}: current outputs, shared forcing, no peak alignment; dashed global peaks',
                  xaxis_title='UTC',yaxis_title='Discharge (m³/s)')
fig.update_xaxes(range=[parent_peak-pd.Timedelta(hours=72),parent_peak+pd.Timedelta(hours=96)])
fig.show()

## Celerity cap: a demonstrated model-choice effect

In 22, 39.7% of parent mean travelled length and 65.3% of segmented-child length use the upper celerity cap. The corresponding child fractions in 41, 46 and 55 are 0%, 1.7% and 0%.

Diagnostic reruns vary the cap in **both parent and child**, retain geometry, forcing, calibrated Manning n and beta, and recompute K/X consistently. Cells are increased where required by the existing grid bound. Calibration is not refitted, so these are sensitivities, not alternative validated models. All runs use the same published 0.5% interval and shared baseline; that baseline remains unchanged. Window boundaries are not reoptimized for these experimental caps.

| Upper cap | Centroid shift | Peak shift |
|---|---:|---:|
| 5 m/s — current | −5.263 h | −5.717 h |
| 10 m/s | −7.855 h | −8.967 h |
| 50 m/s — no reaches hit cap | −8.243 h | −9.450 h |

Thus the cap suppresses roughly three hours of centroid advance in this controlled comparison. It does **not** explain the entire difference from the forest's −13.638-hour prediction.


In [5]:
display(pd.read_csv(OUT/'cap_sensitivity22.csv').query("state=='all_closed_units_collapsed'")
        [['cap_mps','remaining_capped_links','centroid_shift_hours','peak_shift_hours']])
blocks=pd.read_csv(OUT/'block_attribution.csv',dtype={'example':str})
z=blocks[blocks.example==EXAMPLE].sort_values('delta_hours')
display(z[['state','parent_length_km','child_length_km','parent_effective_speed','child_speed','delta_hours','length_term_hours','speed_term_hours']].round(3))
px.bar(z,x='state',y=['length_term_hours','speed_term_hours'],title=f'{EXAMPLE}: block contributions to route-time change',labels={'value':'Hours'}).show()

,cap_mps,remaining_capped_links,centroid_shift_hours,peak_shift_hours
1,5.0,12,-5.263388,-5.716667
3,10.0,3,-7.855201,-8.966667
5,50.0,0,-8.243440,-9.450000


,state,parent_length_km,child_length_km,parent_effective_speed,child_speed,delta_hours,length_term_hours,speed_term_hours
116,CB002_bubble_11,102.571,94.318,4.045,5.000,-1.804,-0.567,-1.237
115,CB001_bubble_9,91.683,81.784,4.593,5.000,-1.002,-0.599,-0.403
118,CB004_bubble_8,11.699,10.735,1.552,2.670,-0.977,-0.173,-0.804
117,CB003_bubble_10,22.530,20.147,3.301,5.000,-0.777,-0.201,-0.576
128,S216_unit_10,11.905,10.954,1.534,2.128,-0.726,-0.172,-0.554
125,S170_unit_6,12.962,11.945,2.066,2.765,-0.543,-0.137,-0.406
123,S049_unit_9,3.516,3.151,1.096,1.535,-0.320,-0.093,-0.228
126,S180_unit_331,7.881,7.421,2.172,2.891,-0.295,-0.059,-0.236
124,S057_unit_5,1.881,1.674,0.824,1.163,-0.234,-0.070,-0.164
120,CB006_bubble_1,6.315,5.602,3.306,5.000,-0.219,-0.060,-0.160


## Why the geometric forest struggles

Size and geometric ratios do not uniquely establish an absolute time scale. For example, edited parent length is 290 km in 22 and 199 km in 55, but the model-derived parent effective speeds inside those regions are 3.26 and 1.00 m/s. The smaller/slower 55 has a much larger time reduction. These speeds depend on slope, depth, Manning n and the cap; they are diagnostic model quantities, not new input predictors.

All 15 segmented cases have zero child route entropy and zero child route-length SD. Those two ratio inputs therefore provide no within-segmented variation. Remaining aggregate predictors omit the calibrated speed scale and its cap, and may lose spatial hydraulic variation. This is a mechanistic hypothesis supported by the routing audit, not proof that any one new geometric variable will solve generalization.

The same 22/46/55 outer models were refitted exactly and reproduced their saved predictions. Co-leaf neighborhoods show 22 is grouped prominently with segmented 41/55/46, which have much larger advances. For 46 and 55, frequently co-located segmented cases include 13/05/45, with smaller advances. Neighborhood statistics use all training rows and original weights; they are **not** bootstrap-aware SHAP or exact influence attribution.

Across these 15 cases, the model-derived route-time change and finite-window centroid shift correlate strongly (Pearson 0.981, Spearman 0.961). This is a descriptive diagnostic using model parameters; it must not be presented as out-of-network predictive performance or inserted into the geometric model as a disguised output-derived predictor.


In [7]:
edited=pd.read_csv(OUT/'edited_domain_summary.csv',dtype={'example':str})
display(edited[edited.example.isin(['22','41','46','55'])][['example','parent_length_km','mean_length_ratio','mean_width_ratio','parent_effective_speed','child_effective_speed','delta_hours','centroid_shift_hours']].round(3))
routing=pd.read_csv(OUT/'routing_summary.csv',dtype={'example':str})
px.scatter(routing,x='delta_time_hours',y='centroid_shift_hours',text='example',
           title='Routing diagnostic, not predictive validation',labels={'delta_time_hours':'Change in mean route time (hours)','centroid_shift_hours':'Centroid shift (hours)'}).show()
leaf=pd.read_parquet(OUT/'leaf_neighborhoods.parquet')
display(leaf[leaf.cohort.eq('segmented')].groupby(['held_example','example']).co_leaf_weight.sum().sort_values(ascending=False).head(15))

,example,parent_length_km,mean_length_ratio,mean_width_ratio,parent_effective_speed,child_effective_speed,delta_hours,centroid_shift_hours
6,22,290.206,0.906,4.792,3.260,4.152,-7.131,-5.263
8,41,264.649,0.888,3.114,1.971,3.203,-16.919,-13.244
10,46,435.613,0.919,2.257,1.191,1.686,-35.634,-23.012
13,55,198.510,0.946,2.460,1.002,1.472,-19.563,-17.279


held_example  example
22            41         0.275320
              55         0.244520
              46         0.212345
46            13         0.174866
55            13         0.173884
46            05         0.160884
55            05         0.160394
46            45         0.124638
55            45         0.123939
22            13         0.086020
46            56         0.084002
55            56         0.083925
22            45         0.083657
55            41         0.070855
46            41         0.070462
Name: co_leaf_weight, dtype: float64

## What this supports next

1. Keep the segmented outcomes: no response-join, sign, baseline or timestep error explains the 22 result.
2. Separate **model-choice sensitivity** (celerity cap, Manning n, depth prescription) from **geometry-only predictor additions**. Do not silently add calibrated speed or child hydrograph quantities to the predictor set.
3. Audit which absolute slope/width descriptors are genuinely available from hydrography, and whether their spatial variation explains the contrasting 22/41/46/55 response scales. A saved model slope or hydraulic head is not automatically an independent hydrographic measurement; provenance must be checked before proposing it as an input.
4. Retain the finite-window response definition. Its interaction with event shape and different equilibrium endpoints is real and can contribute residual variation beyond geometry.

These diagnostics narrow the explanation substantially but do not establish a new final predictor or validate removal of the cap. All experiment files are under `hydrograph_closed_unit_extension_20261019/segmented_diagnostics`; scripts are `investigate_segmented.py`, `check22_routing.py`, and `check22_cap.py` in the corresponding investigations directory. No further model search was performed in this investigation.

## Slope follow-up completed

See [slope provenance and factor attribution](hydrograph_slope_attribution.ipynb). Active parent slopes match the observation-derived robust-profile hydraulic sources in all 15 networks. Slope contributes to the speed contrast of 22 with 41/46, but the 22/55 contrast is mainly associated with prescribed depth and calibrated roughness; 55 has the steeper effective slope. The route-length-weighted mean-slope ratio is exactly redundant with the inverse mean-length ratio for these boundary-head-preserving segmented collapses. Absolute effective parent slope and a separate variability descriptor are candidates; no new predictive performance has been claimed.